# spark-perf-lint — Tier 3 Deep Audit

**Interactive runtime analysis for PySpark performance problems.**

This notebook complements the Tier 1 static linter by executing queries on a
live Spark runtime and measuring *actual* shuffle bytes, spill, skew ratios,
and plan strategies — the ground truth that static analysis can only estimate.

| Section | What it measures |
|---------|------------------|
| 1. Setup | SparkSession · sparkMeasure · helpers |
| 2. Data Generators | Synthetic skewed / join / partition data |
| 3. Physical Plan Analysis | Join strategy · Exchange count · plan diff |
| 4. Partition Analysis | Rows per partition · skew ratio · histogram |
| 5. Shuffle Analysis | Bytes · fetch wait · spill via sparkMeasure |
| 6. AQE Verification | With / without AQE side-by-side |
| 7. Performance Comparison | Bad vs good code, wall-clock + metric table |

> **Tier 3 requires** a live Spark runtime.  
> Run `pip install pyspark sparkmeasure matplotlib` before executing cells.


---
## 1 · Setup — SparkSession & sparkMeasure

The session starts with **AQE disabled** so later sections can observe the
difference when it is toggled on. Adjust `spark.sql.shuffle.partitions` to
match your cluster (200 is the Spark default; lower it for local mode).


In [ ]:
# ── Prerequisites check ───────────────────────────────────────────────────
import importlib, subprocess, sys

_REQUIRED = {
    "pyspark":      "pip install pyspark",
    "sparkmeasure": "pip install sparkmeasure",
    "matplotlib":   "pip install matplotlib",
    "numpy":        "pip install numpy",
}
missing = [pkg for pkg in _REQUIRED if importlib.util.find_spec(pkg) is None]
if missing:
    print('Missing packages — install before continuing:')
    for pkg in missing:
        print(f'  {_REQUIRED[pkg]}')
else:
    print('All prerequisites found ✓')


In [ ]:
import io, re, time, warnings
from contextlib import redirect_stdout
from typing import Any, Callable

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 4)})


In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType, IntegerType, LongType, StringType,
    StructField, StructType,
)

spark = (
    SparkSession.builder
    .appName("spark-perf-lint:deep-audit")
    # AQE intentionally OFF — toggled in Section 6
    .config("spark.sql.adaptive.enabled",                    "false")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "false")
    .config("spark.sql.adaptive.skewJoin.enabled",           "false")
    .config("spark.sql.shuffle.partitions",                  "50")
    .config("spark.ui.enabled",                              "true")
    .config("spark.driver.memory",                          "2g")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

print(f'Spark version : {spark.version}')
print(f'Master        : {spark.sparkContext.master}')
ui = spark.sparkContext.uiWebUrl
if ui:
    print(f'Spark UI      : {ui}')


In [ ]:
# ── sparkMeasure (optional but recommended for §5) ─────────────────────────
HAS_SPARKMEASURE = False
try:
    from sparkmeasure import StageMetrics, TaskMetrics  # type: ignore[import]
    HAS_SPARKMEASURE = True
    print('sparkMeasure loaded ✓')
except ImportError:
    print('⚠  sparkMeasure not found — section 5 will use Spark accumulators.')
    print('   Install with: pip install sparkmeasure')


In [ ]:
# ── Shared utilities ────────────────────────────────────────────────────────

def _capture_explain(df: DataFrame, mode: str = 'formatted') -> str:
    """Return df.explain() output as a string instead of printing it."""
    buf = io.StringIO()
    with redirect_stdout(buf):
        df.explain(mode=mode)
    return buf.getvalue()


def print_metrics_table(rows: list[dict], title: str = '') -> None:
    """Pretty-print a list of metric dicts as an aligned table."""
    if not rows:
        return
    if title:
        print(f'\n{title}')
        print('─' * 70)
    keys = list(rows[0].keys())
    widths = {k: max(len(k), max(len(str(r.get(k, ''))) for r in rows)) for k in keys}
    header = '  '.join(k.ljust(widths[k]) for k in keys)
    print(header)
    print('  '.join('─' * widths[k] for k in keys))
    for row in rows:
        print('  '.join(str(row.get(k, '')).ljust(widths[k]) for k in keys))


print('Utilities loaded ✓')


---
## 2 · Synthetic Data Generators

Three generators produce DataFrames that trigger known Spark anti-patterns:

- **`gen_uniform`** — evenly distributed keys; baseline for comparisons.
- **`gen_skewed`** — one *hot key* carries `skew_pct`% of all rows; triggers
  skewed joins and uneven partition loading.
- **`gen_join_pair`** — (large fact, small dimension) table pair; useful for
  demonstrating broadcast vs sort-merge join strategy choice.
- **`gen_partition_data`** — explicit row-to-partition mapping to reproduce
  small-files or over-partitioned scenarios.


In [ ]:
def gen_uniform(
    n_rows: int = 500_000,
    n_keys: int = 1_000,
    seed: int = 42,
) -> DataFrame:
    """Uniform key distribution: ~n_rows / n_keys rows per key.

    Args:
        n_rows:  Total number of rows to generate.
        n_keys:  Distinct key cardinality.
        seed:    Random seed for reproducibility.

    Returns:
        DataFrame with columns: join_key (string), value (long), amount (double).
    """
    return (
        spark.range(n_rows, numPartitions=20)
        .withColumn('join_key', (F.col('id') % n_keys).cast('string'))
        .withColumn('amount',   (F.rand(seed) * 1000).cast(DoubleType()))
        .withColumnRenamed('id', 'value')
    )


def gen_skewed(
    n_rows: int = 500_000,
    hot_key: str = 'HOT',
    skew_pct: float = 0.80,
    n_normal_keys: int = 500,
    seed: int = 42,
) -> DataFrame:
    """Heavily skewed key distribution.

    `skew_pct` of all rows map to a single `hot_key`; the remainder
    are spread evenly across `n_normal_keys` distinct keys.

    Args:
        n_rows:        Total row count.
        hot_key:       The key value that receives the hot partition.
        skew_pct:      Fraction of rows assigned to the hot key (0–1).
        n_normal_keys: Distinct key count for the non-hot rows.
        seed:          Random seed.

    Returns:
        DataFrame with columns: join_key (string), value (long), amount (double).
    """
    hot_count    = int(n_rows * skew_pct)
    normal_count = n_rows - hot_count

    hot = (
        spark.range(hot_count, numPartitions=5)
        .withColumn('join_key', F.lit(hot_key))
        .withColumn('amount',   (F.rand(seed) * 1000).cast(DoubleType()))
        .withColumnRenamed('id', 'value')
    )
    normal = (
        spark.range(normal_count, numPartitions=15)
        .withColumn('join_key', (F.col('id') % n_normal_keys).cast('string'))
        .withColumn('amount',   (F.rand(seed + 1) * 1000).cast(DoubleType()))
        .withColumnRenamed('id', 'value')
    )
    return hot.unionAll(normal)


def gen_join_pair(
    fact_rows: int = 1_000_000,
    dim_rows:  int = 5_000,
    seed: int = 42,
) -> tuple[DataFrame, DataFrame]:
    """Fact + dimension table pair for join strategy benchmarks.

    The dimension table is intentionally small so that broadcast join
    is a valid (and much cheaper) strategy compared to sort-merge join.

    Returns:
        (fact_df, dim_df) — fact has join_key, amount, ts;
        dim has join_key, category, discount.
    """
    fact = (
        spark.range(fact_rows, numPartitions=20)
        .withColumn('join_key', (F.col('id') % dim_rows).cast('string'))
        .withColumn('amount',   (F.rand(seed) * 1000).cast(DoubleType()))
        .withColumn('ts',       (F.col('id') % 86400).cast(LongType()))
        .withColumnRenamed('id', 'fact_id')
    )
    dim = (
        spark.range(dim_rows, numPartitions=2)
        .withColumn('join_key',  F.col('id').cast('string'))
        .withColumn('category',  (F.col('id') % 10).cast('string'))
        .withColumn('discount',  (F.rand(seed + 2) * 0.5).cast(DoubleType()))
        .drop('id')
    )
    return fact, dim


def gen_partition_data(
    n_rows: int = 200_000,
    n_partitions: int = 50,
    empty_fraction: float = 0.0,
) -> DataFrame:
    """DataFrame with controllable partition count.

    When `empty_fraction > 0`, that fraction of partition slots will hold
    no rows, simulating the small-files / over-partitioned scenario.

    Returns:
        DataFrame with columns: partition_id (int), value (long).
    """
    df = spark.range(n_rows, numPartitions=n_partitions)
    df = df.withColumn('partition_id', F.spark_partition_id())
    if empty_fraction > 0:
        keep = int(n_partitions * (1 - empty_fraction))
        df = df.filter(F.col('partition_id') < keep)
    return df


# Quick smoke test
print('Generating sample DataFrames…')
_u = gen_uniform(10_000)
_s = gen_skewed(10_000)
_f, _d = gen_join_pair(50_000, 500)
print(f'  uniform      rows={_u.count():,}   partitions={_u.rdd.getNumPartitions()}')
print(f'  skewed       rows={_s.count():,}   partitions={_s.rdd.getNumPartitions()}')
print(f'  fact         rows={_f.count():,}  partitions={_f.rdd.getNumPartitions()}')
print(f'  dimension    rows={_d.count():,}     partitions={_d.rdd.getNumPartitions()}')
print('Generators ready ✓')


In [ ]:
# ── Visualise key distribution (uniform vs skewed) ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (label, df) in zip(axes, [('Uniform', _u), ('Skewed (80 % hot key)', _s)]):
    counts = (
        df.groupBy('join_key')
          .agg(F.count('*').alias('cnt'))
          .orderBy(F.col('cnt').desc())
          .limit(30)
          .toPandas()
    )
    ax.bar(range(len(counts)), counts['cnt'], color='steelblue', width=0.8)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel('Key rank (top 30)')
    ax.set_ylabel('Row count')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))

plt.suptitle('Key distribution — uniform vs skewed', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


---
## 3 · Physical Plan Analysis

Spark's physical plan (`df.explain(mode='formatted')`) reveals the actual
join strategy Catalyst chose and how many shuffles (Exchanges) are required.

Key patterns to detect:

| Pattern in plan | Meaning |
|---|---|
| `BroadcastHashJoin` | Small table broadcast — no shuffle ✓ |
| `SortMergeJoin` | Both sides shuffled and sorted — expensive |
| `ShuffleHashJoin` | One side shuffled into hash table |
| `CartesianProduct` | Cross join — O(N×M) — almost always a bug |
| `Exchange hashpartitioning` | Shuffle boundary — counts matter |
| `AQEShuffleRead` | AQE coalesced post-shuffle partitions ✓ |


In [ ]:
# ── Plan analysis helpers ───────────────────────────────────────────────────

_JOIN_STRATEGIES = [
    'BroadcastHashJoin',
    'SortMergeJoin',
    'ShuffleHashJoin',
    'BroadcastNestedLoopJoin',
    'CartesianProduct',
]


def analyze_plan(df: DataFrame, label: str = '') -> dict:
    """Parse the physical plan and return a structured summary.

    Args:
        df:    DataFrame whose plan to analyse.
        label: Optional description for display.

    Returns:
        Dict with keys: label, join_strategies, exchange_count,
        aqe_reads, plan_preview.
    """
    plan = _capture_explain(df, mode='formatted')

    strategies = sorted(
        {s for s in _JOIN_STRATEGIES if s in plan}
    )
    exchanges  = len(re.findall(r'Exchange ', plan))
    aqe_reads  = len(re.findall(r'AQEShuffleRead', plan))

    return {
        'label':          label or '(unlabelled)',
        'join_strategies': ', '.join(strategies) if strategies else 'none',
        'exchange_count':  exchanges,
        'aqe_reads':       aqe_reads,
        'plan_preview':    plan[:600].replace('\n', ' | ')[:120] + '…',
    }


def diff_plans(plan_a: str, plan_b: str, label_a: str = 'A', label_b: str = 'B') -> None:
    """Print a side-by-side exchange/strategy diff between two plans."""
    def _extract(p: str) -> dict:
        return {
            'strategies': {s for s in _JOIN_STRATEGIES if s in p},
            'exchanges':  len(re.findall(r'Exchange ', p)),
            'aqe_reads':  len(re.findall(r'AQEShuffleRead', p)),
        }
    a, b = _extract(plan_a), _extract(plan_b)
    print(f'{'Metric':<22} {label_a:<28} {label_b}')
    print('─' * 70)
    print(f'{'Join strategies':<22} {str(a["strategies"]):<28} {b["strategies"]}')
    print(f'{'Exchange count':<22} {a["exchanges"]:<28} {b["exchanges"]}')
    print(f'{'AQEShuffleRead nodes':<22} {a["aqe_reads"]:<28} {b["aqe_reads"]}')


print('Plan analysis helpers loaded ✓')


In [ ]:
# ── Demonstrate: sort-merge join vs broadcast join ──────────────────────────
fact_df, dim_df = gen_join_pair(200_000, 3_000)

# No hint — Catalyst decides (usually SortMergeJoin for these sizes in local mode)
no_hint_join = fact_df.join(dim_df, 'join_key')

# Explicit broadcast hint on the small dimension table
broadcast_join = fact_df.join(F.broadcast(dim_df), 'join_key')

results = [
    analyze_plan(no_hint_join,   'No hint (auto)'),
    analyze_plan(broadcast_join, 'Broadcast hint'),
]
print_metrics_table(
    [{k: v for k, v in r.items() if k != 'plan_preview'} for r in results],
    title='Join strategy comparison',
)

print('\n── Plan diff ──────────────────────────────────────────────────────────')
diff_plans(
    _capture_explain(no_hint_join),
    _capture_explain(broadcast_join),
    label_a='No hint',
    label_b='broadcast(dim)',
)


In [ ]:
# ── Print full physical plan for the broadcast join ─────────────────────────
print(_capture_explain(broadcast_join, mode='formatted')[:2500])


---
## 4 · Partition Analysis

Uneven partition loading is the root cause of **task stragglers** — one task
running 10× longer than others while the rest of the cluster idles.

Key metrics:

- **Skew ratio** = `max_rows / mean_rows` — values > 3 are concerning;
  values > 10 cause noticeable straggler tasks.
- **Empty partitions** — drive overhead without work; often left by
  `coalesce` or aggressive `filter` operations.
- **Standard deviation / IQR** — characterise spread beyond the max.


In [ ]:
def partition_stats(df: DataFrame) -> dict:
    """Compute per-partition row counts and skew statistics.

    Uses mapPartitionsWithIndex to count rows without a full shuffle.

    Returns:
        Dict with n_partitions, total_rows, min_rows, max_rows,
        mean_rows, median_rows, std_rows, skew_ratio, empty_count,
        and per_partition (dict of partition_id → row_count).
    """
    per_part: dict[int, int] = dict(
        df.rdd
          .mapPartitionsWithIndex(lambda i, it: [(i, sum(1 for _ in it))])
          .collect()
    )
    vals = np.array(list(per_part.values()), dtype=float)
    mean = float(vals.mean()) if len(vals) else 0.0
    return {
        'n_partitions': len(vals),
        'total_rows':   int(vals.sum()),
        'min_rows':     int(vals.min()) if len(vals) else 0,
        'max_rows':     int(vals.max()) if len(vals) else 0,
        'mean_rows':    round(mean, 1),
        'median_rows':  round(float(np.median(vals)), 1),
        'std_rows':     round(float(vals.std()), 1),
        'skew_ratio':   round(float(vals.max()) / mean, 2) if mean else 0.0,
        'empty_count':  int((vals == 0).sum()),
        'per_partition': per_part,
    }


def plot_partition_histogram(
    stats_list: list[dict],
    labels: list[str],
    bins: int = 40,
) -> None:
    """Plot row-count histograms for one or more partition stat dicts."""
    n = len(stats_list)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 4), squeeze=False)
    colours = ['steelblue', 'tomato', 'seagreen', 'goldenrod']

    for ax, stats, label, colour in zip(axes[0], stats_list, labels, colours):
        data = list(stats['per_partition'].values())
        ax.hist(data, bins=bins, color=colour, edgecolor='white', linewidth=0.4)
        ax.axvline(stats['mean_rows'],   color='black',  ls='--', lw=1.5, label=f'mean={stats["mean_rows"]:.0f}')
        ax.axvline(stats['max_rows'],    color='crimson', ls=':',  lw=1.5, label=f'max={stats["max_rows"]}')
        ax.set_title(
            f'{label}\nskew_ratio={stats["skew_ratio"]}  '
            f'empty={stats["empty_count"]}',
            fontsize=10, fontweight='bold',
        )
        ax.set_xlabel('Rows per partition')
        ax.set_ylabel('Partition count')
        ax.legend(fontsize=8)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))

    plt.suptitle('Partition row-count distribution', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()


print('Partition analysis helpers loaded ✓')


In [ ]:
# ── Analyse uniform vs skewed DataFrames after a shuffle ───────────────────
# A groupBy triggers a shuffle; the post-shuffle partition distribution
# reveals whether data is balanced across reducer tasks.

uniform_df = gen_uniform(100_000, n_keys=200)
skewed_df  = gen_skewed(100_000, skew_pct=0.75)

# Force a shuffle so partition stats reflect the post-shuffle distribution
uniform_agg = uniform_df.groupBy('join_key').agg(F.sum('amount').alias('total'))
skewed_agg  = skewed_df.groupBy('join_key').agg(F.sum('amount').alias('total'))

print('Computing partition stats (this triggers the shuffle)…')
s_uniform = partition_stats(uniform_agg)
s_skewed  = partition_stats(skewed_agg)

print_metrics_table(
    [
        {k: v for k, v in s_uniform.items() if k != 'per_partition'},
        {k: v for k, v in s_skewed.items()  if k != 'per_partition'},
    ],
    title='Partition statistics — post groupBy shuffle',
)


In [ ]:
plot_partition_histogram(
    [s_uniform, s_skewed],
    ['Uniform distribution', 'Skewed distribution (75 % hot key)'],
)


In [ ]:
# ── Over-partitioned scenario: many empty partitions ───────────────────────
# Simulate a filter that removes 70 % of rows after repartition,
# leaving most task slots idle.

dense = gen_partition_data(100_000, n_partitions=50, empty_fraction=0.0)
sparse = gen_partition_data(100_000, n_partitions=50, empty_fraction=0.7)

s_dense  = partition_stats(dense)
s_sparse = partition_stats(sparse)

print_metrics_table(
    [
        {'scenario': 'dense  (0 % empty)', **{k: v for k, v in s_dense.items()  if k != 'per_partition'}},
        {'scenario': 'sparse (70 % empty)', **{k: v for k, v in s_sparse.items() if k != 'per_partition'}},
    ],
    title='Dense vs sparse partition occupancy',
)


---
## 5 · Shuffle Analysis

Shuffle metrics surface costs invisible in the query plan:

| Metric | What it reveals |
|---|---|
| `shuffleWriteBytes` | Data serialised and written by map tasks |
| `shuffleReadBytes` | Data fetched over the network by reduce tasks |
| `shuffleFetchWaitTime` | Reducer blocked waiting for map output |
| `memoryBytesSpilled` | Data that didn't fit in executor memory |
| `diskBytesSpilled` | Same data written to disk (very expensive) |

When `sparkMeasure` is unavailable, the section falls back to
`SparkContext.statusTracker()` accumulators (less detail but no extra JARs).


In [ ]:
def run_with_metrics(
    label: str,
    fn: Callable[[], Any],
) -> dict:
    """Execute fn() and return a metrics dict.

    Uses sparkMeasure StageMetrics when available; falls back to
    wall-clock timing only when the package is absent.

    Returns:
        Dict with label, elapsed_s, shuffle_write_mb, shuffle_read_mb,
        fetch_wait_ms, mem_spill_mb, disk_spill_mb, task_count.
    """
    if HAS_SPARKMEASURE:
        sm = StageMetrics(spark)
        sm.begin()
        t0 = time.perf_counter()
        result = fn()
        elapsed = time.perf_counter() - t0
        sm.end()
        try:
            # aggregate_stagemetrics() returns a pandas DataFrame
            agg = sm.aggregate_stagemetrics().iloc[0].to_dict()
        except Exception:
            agg = {}
        return {
            'label':          label,
            'elapsed_s':      round(elapsed, 3),
            'shuffle_write_mb': round(agg.get('shuffleWriteBytes', 0) / 1e6, 2),
            'shuffle_read_mb':  round(agg.get('shuffleReadBytes',  0) / 1e6, 2),
            'fetch_wait_ms':    round(agg.get('shuffleFetchWaitTime', 0) / 1e6, 2),
            'mem_spill_mb':     round(agg.get('memoryBytesSpilled', 0) / 1e6, 2),
            'disk_spill_mb':    round(agg.get('diskBytesSpilled',   0) / 1e6, 2),
            'task_count':       int(agg.get('numTasks', 0)),
            '_result':          result,
        }
    else:
        # Fallback: wall-clock only
        t0 = time.perf_counter()
        result = fn()
        elapsed = time.perf_counter() - t0
        return {
            'label':          label,
            'elapsed_s':      round(elapsed, 3),
            'shuffle_write_mb': 'n/a (no sparkMeasure)',
            'shuffle_read_mb':  'n/a',
            'fetch_wait_ms':    'n/a',
            'mem_spill_mb':     'n/a',
            'disk_spill_mb':    'n/a',
            'task_count':       'n/a',
            '_result':          result,
        }


print('run_with_metrics() loaded ✓')
print(f'sparkMeasure available: {HAS_SPARKMEASURE}')


In [ ]:
# ── Shuffle bytes: broadcast join vs sort-merge join ───────────────────────
fact_df, dim_df = gen_join_pair(300_000, 2_000)
fact_df.cache(); dim_df.cache()
fact_df.count(); dim_df.count()  # materialise caches
print('Tables cached.')

smj_metrics = run_with_metrics(
    'SortMergeJoin (no hint)',
    lambda: fact_df.join(dim_df, 'join_key').agg(F.sum('amount')).collect(),
)
bcj_metrics = run_with_metrics(
    'BroadcastHashJoin (hint)',
    lambda: fact_df.join(F.broadcast(dim_df), 'join_key').agg(F.sum('amount')).collect(),
)

display_cols = ['label', 'elapsed_s', 'shuffle_write_mb', 'shuffle_read_mb',
                'fetch_wait_ms', 'mem_spill_mb', 'task_count']
print_metrics_table(
    [{k: v for k, v in m.items() if k in display_cols}
     for m in [smj_metrics, bcj_metrics]],
    title='Shuffle metrics — SortMergeJoin vs BroadcastHashJoin',
)

fact_df.unpersist(); dim_df.unpersist()


In [ ]:
# ── Visualise shuffle bytes comparison ─────────────────────────────────────
metrics_list = [smj_metrics, bcj_metrics]
labels = [m['label'] for m in metrics_list]
elapsed = [m['elapsed_s'] for m in metrics_list]

has_mb = all(isinstance(m['shuffle_write_mb'], (int, float)) for m in metrics_list)
n_plots = 3 if has_mb else 1
fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4))
if n_plots == 1:
    axes = [axes]

colours = ['steelblue', 'tomato']
axes[0].bar(labels, elapsed, color=colours)
axes[0].set_title('Wall-clock time (seconds)', fontweight='bold')
axes[0].set_ylabel('seconds')

if has_mb:
    write_mb = [m['shuffle_write_mb'] for m in metrics_list]
    read_mb  = [m['shuffle_read_mb']  for m in metrics_list]
    axes[1].bar(labels, write_mb, color=colours)
    axes[1].set_title('Shuffle write (MB)', fontweight='bold')
    axes[1].set_ylabel('MB')
    axes[2].bar(labels, read_mb, color=colours)
    axes[2].set_title('Shuffle read (MB)', fontweight='bold')
    axes[2].set_ylabel('MB')

plt.suptitle('Join strategy — shuffle cost comparison', fontsize=12)
plt.tight_layout()
plt.show()


---
## 6 · AQE Verification — with vs without Adaptive Query Execution

Adaptive Query Execution (Spark 3.0+) has three main features:

| AQE feature | What it does | Config key |
|---|---|---|
| Coalesce partitions | Merges small post-shuffle partitions | `spark.sql.adaptive.coalescePartitions.enabled` |
| Skew join | Splits skewed map partitions before the join | `spark.sql.adaptive.skewJoin.enabled` |
| Dynamic broadcast | Upgrades SMJ→BHJ at runtime if size fits | `spark.sql.adaptive.enabled` |

This section runs the **same query** with AQE off and on, captures metrics,
and extracts the post-shuffle partition counts from the physical plan to show
coalescing in action.


In [ ]:
def set_aqe(enabled: bool) -> None:
    """Toggle all AQE features on or off on the current SparkSession."""
    v = str(enabled).lower()
    spark.conf.set('spark.sql.adaptive.enabled',                    v)
    spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', v)
    spark.conf.set('spark.sql.adaptive.skewJoin.enabled',           v)
    state = 'ON ✓' if enabled else 'OFF'
    print(f'AQE {state}')


def aqe_partition_count(df: DataFrame) -> int:
    """Return the number of output partitions after the query executes."""
    # Trigger execution and inspect the post-execution partition count
    return df.rdd.getNumPartitions()


print('AQE helpers loaded ✓')


In [ ]:
# ── Benchmark: skewed join with AQE off vs on ─────────────────────────────
# Use a skewed fact table joined against a dimension table.
# AQE skew-join splits the hot partition; AQE coalesce merges empty output partitions.

fact_df, dim_df = gen_join_pair(500_000, 3_000)
skewed_fact = gen_skewed(500_000, skew_pct=0.85)

aqe_results = []

for aqe_on in [False, True]:
    set_aqe(aqe_on)

    def _run_join():
        joined = (
            skewed_fact
            .join(dim_df, 'join_key')
            .groupBy('join_key')
            .agg(F.sum('amount').alias('total'))
        )
        count = joined.count()
        n_partitions = aqe_partition_count(joined)
        return count, n_partitions

    label = f'AQE {"ON" if aqe_on else "OFF"}'
    m = run_with_metrics(label, _run_join)
    count, n_parts = m.pop('_result')
    m['output_partitions'] = n_parts
    m['result_rows']       = count
    aqe_results.append(m)
    print(f'  {label}: {count} rows, {n_parts} output partitions, {m["elapsed_s"]}s')

# Restore AQE off so subsequent sections start from a known baseline
set_aqe(False)


In [ ]:
display_cols = ['label', 'elapsed_s', 'shuffle_write_mb', 'shuffle_read_mb',
                'output_partitions', 'result_rows', 'task_count']
print_metrics_table(
    [{k: v for k, v in r.items() if k in display_cols} for r in aqe_results],
    title='AQE OFF vs ON — skewed join benchmark',
)


In [ ]:
# ── Plan diff: AQE OFF vs AQE ON ──────────────────────────────────────────
set_aqe(False)
plan_aqe_off = _capture_explain(
    skewed_fact.join(dim_df, 'join_key').groupBy('join_key').agg(F.sum('amount'))
)

set_aqe(True)
plan_aqe_on = _capture_explain(
    skewed_fact.join(dim_df, 'join_key').groupBy('join_key').agg(F.sum('amount'))
)

set_aqe(False)  # restore

print('── Physical plan diff: AQE OFF vs AQE ON ─────────────────────────────────')
diff_plans(plan_aqe_off, plan_aqe_on, label_a='AQE OFF', label_b='AQE ON')

print('\n── Key lines from AQE ON plan (AQEShuffleRead) ───────────────────────────')
for line in plan_aqe_on.splitlines():
    if any(kw in line for kw in ['AQEShuffleRead', 'CustomShuffleReader', 'skewJoin']):
        print(' ', line.strip())


In [ ]:
# ── Visualise AQE improvement ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

labels   = [r['label'] for r in aqe_results]
colours  = ['tomato', 'seagreen']

axes[0].bar(labels, [r['elapsed_s'] for r in aqe_results], color=colours)
axes[0].set_title('Wall-clock time', fontweight='bold')
axes[0].set_ylabel('seconds')

axes[1].bar(labels, [r['output_partitions'] for r in aqe_results], color=colours)
axes[1].set_title('Output partitions after coalesce', fontweight='bold')
axes[1].set_ylabel('partition count')

plt.suptitle('AQE impact on skewed join', fontsize=12)
plt.tight_layout()
plt.show()


---
## 7 · Performance Comparison — Bad vs Good Code Patterns

Head-to-head benchmarks for the most impactful anti-patterns detected by
the Tier 1 static linter.  Each pair runs **identical logic** — only the
implementation pattern changes.

| # | Anti-pattern (bad) | Best practice (good) | Expected impact |
|---|---|---|---|
| A | `groupByKey` (RDD) | `reduceByKey` / DataFrame `groupBy+agg` | 2–5× shuffle reduction |
| B | `.join()` no hint | `.join(broadcast(dim))` | Eliminates shuffle on dim side |
| C | `repartition(1)` before write | `coalesce(n)` or natural partitions | Avoids full reshuffle |
| D | `collect()` before groupBy | Push down aggregation | Removes driver bottleneck |


In [ ]:
# ── Shared setup for comparison benchmarks ────────────────────────────────
set_aqe(False)  # keep AQE off for fair, deterministic comparison

BENCH_ROWS = 300_000
bench_df   = gen_skewed(BENCH_ROWS, skew_pct=0.4).cache()
bench_df.count()  # materialise
_, dim_small = gen_join_pair(BENCH_ROWS, dim_rows=500)
dim_small = dim_small.cache()
dim_small.count()
print(f'Benchmark DataFrames ready: fact={BENCH_ROWS:,} rows, dim=500 rows')


In [ ]:
# ══ Benchmark A: groupByKey (RDD) vs groupBy+agg (DataFrame) ══════════════
print('Benchmark A: group-by aggregation pattern')

def _bad_groupby():
    # Anti-pattern: groupByKey collects ALL values per key to a single partition
    # before applying the aggregation — maximum shuffle data.
    return (
        bench_df.rdd
        .map(lambda r: (r['join_key'], r['amount']))
        .groupByKey()
        .mapValues(sum)
        .count()
    )

def _good_groupby():
    # Best practice: combine on the map side before shuffle
    return (
        bench_df
        .groupBy('join_key')
        .agg(F.sum('amount').alias('total'))
        .count()
    )

m_bad_gb  = run_with_metrics('A. groupByKey (RDD)',    _bad_groupby)
m_good_gb = run_with_metrics('A. groupBy+agg (DF)',    _good_groupby)
print(f'  Bad  → {m_bad_gb["elapsed_s"]}s  |  Good → {m_good_gb["elapsed_s"]}s')


In [ ]:
# ══ Benchmark B: SortMergeJoin vs BroadcastHashJoin ═══════════════════════
print('Benchmark B: join strategy')

def _bad_join():
    # Anti-pattern: no hint on a small dim table → Catalyst may choose SMJ
    return bench_df.join(dim_small, 'join_key').agg(F.sum('amount')).collect()

def _good_join():
    # Best practice: explicit broadcast hint eliminates the shuffle on dim_small
    return bench_df.join(F.broadcast(dim_small), 'join_key').agg(F.sum('amount')).collect()

m_bad_join  = run_with_metrics('B. No hint (SMJ)',      _bad_join)
m_good_join = run_with_metrics('B. broadcast(dim)',     _good_join)
print(f'  Bad  → {m_bad_join["elapsed_s"]}s  |  Good → {m_good_join["elapsed_s"]}s')


In [ ]:
# ══ Benchmark C: repartition(1) vs coalesce(n) ════════════════════════════
print('Benchmark C: repartition(1) vs coalesce — write simulation')

import tempfile, os
_tmpdir = tempfile.mkdtemp()

def _bad_repartition():
    # Anti-pattern: full shuffle to a single partition before writing
    path = os.path.join(_tmpdir, 'bad_repart')
    bench_df.repartition(1).write.mode('overwrite').parquet(path)
    return spark.read.parquet(path).count()

def _good_coalesce():
    # Best practice: coalesce avoids the full shuffle
    path = os.path.join(_tmpdir, 'good_coalesce')
    bench_df.coalesce(4).write.mode('overwrite').parquet(path)
    return spark.read.parquet(path).count()

m_bad_rep  = run_with_metrics('C. repartition(1)',  _bad_repartition)
m_good_rep = run_with_metrics('C. coalesce(4)',      _good_coalesce)
print(f'  Bad  → {m_bad_rep["elapsed_s"]}s  |  Good → {m_good_rep["elapsed_s"]}s')


In [ ]:
# ══ Benchmark D: collect-then-aggregate vs push-down aggregation ══════════
print('Benchmark D: driver-side aggregation vs Spark-side')

def _bad_collect():
    # Anti-pattern: collect all rows to driver, then aggregate in Python
    rows = bench_df.collect()
    return sum(r['amount'] for r in rows)

def _good_agg():
    # Best practice: push the aggregation into Spark; only the scalar returns
    return bench_df.agg(F.sum('amount')).collect()[0][0]

m_bad_collect  = run_with_metrics('D. collect() + Python sum', _bad_collect)
m_good_collect = run_with_metrics('D. Spark agg (push-down)',  _good_agg)
print(f'  Bad  → {m_bad_collect["elapsed_s"]}s  |  Good → {m_good_collect["elapsed_s"]}s')


In [ ]:
# ── Side-by-side metrics table ────────────────────────────────────────────
all_metrics = [
    m_bad_gb, m_good_gb,
    m_bad_join, m_good_join,
    m_bad_rep, m_good_rep,
    m_bad_collect, m_good_collect,
]
display_cols = ['label', 'elapsed_s', 'shuffle_write_mb', 'shuffle_read_mb',
                'mem_spill_mb', 'task_count']
print_metrics_table(
    [{k: v for k, v in m.items() if k in display_cols and k != '_result'}
     for m in all_metrics],
    title='All benchmarks — bad vs good patterns',
)


In [ ]:
# ── Bar chart: elapsed time across all benchmarks ─────────────────────────
pairs = [
    ('A. groupBy', m_bad_gb,  m_good_gb),
    ('B. join',    m_bad_join, m_good_join),
    ('C. write',   m_bad_rep,  m_good_rep),
    ('D. collect', m_bad_collect, m_good_collect),
]

x = np.arange(len(pairs))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, [p[1]['elapsed_s'] for p in pairs], width=w,
       color='tomato',   label='Anti-pattern (bad)', alpha=0.9)
ax.bar(x + w/2, [p[2]['elapsed_s'] for p in pairs], width=w,
       color='seagreen', label='Best practice (good)', alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels([p[0] for p in pairs], fontsize=11)
ax.set_ylabel('Elapsed time (seconds)')
ax.set_title('Anti-pattern vs best practice — wall-clock time', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

# Annotate speedup
for i, (_, bad, good) in enumerate(pairs):
    speedup = bad['elapsed_s'] / good['elapsed_s'] if good['elapsed_s'] > 0 else 0
    ax.annotate(
        f'{speedup:.1f}×',
        xy=(x[i], max(bad['elapsed_s'], good['elapsed_s']) + 0.01),
        ha='center', fontsize=9, color='navy', fontweight='bold',
    )

plt.tight_layout()
plt.show()


In [ ]:
# ── Shuffle bytes bar chart (when sparkMeasure available) ─────────────────
has_mb = all(
    isinstance(m.get('shuffle_write_mb'), (int, float))
    for m in all_metrics
)

if has_mb:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    colours_bad  = ['#d62728', '#d62728', '#d62728', '#d62728']
    colours_good = ['#2ca02c', '#2ca02c', '#2ca02c', '#2ca02c']

    for ax, metric_key, title in [
        (axes[0], 'shuffle_write_mb', 'Shuffle write (MB)'),
        (axes[1], 'shuffle_read_mb',  'Shuffle read (MB)'),
    ]:
        bad_vals  = [p[1].get(metric_key, 0) for p in pairs]
        good_vals = [p[2].get(metric_key, 0) for p in pairs]
        ax.bar(x - w/2, bad_vals,  width=w, color='tomato',   label='bad',  alpha=0.9)
        ax.bar(x + w/2, good_vals, width=w, color='seagreen', label='good', alpha=0.9)
        ax.set_xticks(x)
        ax.set_xticklabels([p[0] for p in pairs])
        ax.set_title(title, fontweight='bold')
        ax.set_ylabel('MB')
        ax.legend()

    plt.suptitle('Shuffle cost — bad vs good patterns', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('Shuffle byte charts require sparkMeasure — install and re-run.')


---
## Summary & Next Steps

### Findings from this audit

| Section | Key signal | Tier 1 rule |
|---------|------------|-------------|
| 3 — Plans | SortMergeJoin on small table | SPL-D03-002 |
| 4 — Partitions | Skew ratio > 3× | SPL-D05-001/003 |
| 5 — Shuffle | Shuffle write ≫ broadcast join | SPL-D02-001/004 |
| 6 — AQE | Partition count unchanged with AQE off | SPL-D08-001/003 |
| 7 — Benchmarks | groupByKey 2–5× slower than agg | SPL-D02-002 |

### Recommended actions

1. **Enable AQE** (`spark.sql.adaptive.enabled = true`) in all production
   sessions — zero code change, significant partition coalescing.
2. **Add broadcast hints** for dimension tables < 10 MB
   (`spark.sql.autoBroadcastJoinThreshold` controls the auto threshold).
3. **Replace `groupByKey`** with `reduceByKey` (RDD) or `groupBy().agg()`
   (DataFrame) everywhere — the shuffle reduction is consistently 2–5×.
4. **Never `collect()` before aggregating** — push all aggregations into Spark
   and only pull the final scalar to the driver.
5. **Profile before and after** each change with `run_with_metrics()` to
   confirm the expected improvement in your cluster environment.

### Exporting results

```python
# Save the benchmark table to JSON for inclusion in a report
import json
safe = [{k: v for k, v in m.items() if k != '_result'} for m in all_metrics]
with open('deep_audit_results.json', 'w') as f:
    json.dump(safe, f, indent=2, default=str)
```


In [ ]:
# ── Clean up ──────────────────────────────────────────────────────────────
bench_df.unpersist()
dim_small.unpersist()

import shutil
shutil.rmtree(_tmpdir, ignore_errors=True)

print('Benchmark DataFrames unpersisted.')
print('Temporary write directory cleaned up.')
print('Deep audit complete ✓')

# Uncomment to stop the session when done:
# spark.stop()
